# Natural Language Processing Pipeline
## Task 2: Model Training


Environment: Python 3 and Jupyter notebook

Libraries used: please include all the libraries you used in your assignment, e.g.,:
* pandas
* numpy
* CountVectorizer
* Gensim
* TfidfVectorizer
* LogisticRegression
* KFold
* f1_score
* Chain
* RegexpTokenizer
* sent_tokenize
* FreqDist



## Introduction
- Generating feature representation for 'Processed Review Text', including Count Vector, Unweighted Vector, Weighted Vector (TF-IDF). 
- Preprocessed 'Title' column
- Generating feature representation for 'Title', including Count Vector, Unweighted Vector, Weighted Vector (TF-IDF). 
- Integrating 'Processed Review Text' and 'Processed Title'
- Generating feature representation for 'Title', including Count Vector, Unweighted Vector, Weighted Vector (TF-IDF). 
- Doing performance classification for 'Processed Review Text', 'Processed Title', and 'Processed Title + Review'
- Make comparison of the results


## Importing libraries 

In [8]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import gensim.downloader as api
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score
from itertools import chain
from nltk import RegexpTokenizer
from nltk.tokenize import sent_tokenize
from nltk.probability import FreqDist




## Task 2. Generating Feature Representations for Clothing Items Reviews

#### Count Vector

In [9]:
# load vocab
vocab_dict = {}
with open('vocab.txt', 'r') as f:
    for line in f.read().splitlines():
        word, idx = line.split(':')
        vocab_dict[word] = int(idx)

print(f"Vocab length: {len(vocab_dict)}")
print("Preview:")
for word, idx in list(vocab_dict.items())[:10]:
    print(f"{word}:{idx}")


Vocab length: 7529
Preview:
a-cup:0
a-flutter:1
a-frame:2
a-kind:3
a-line:4
a-lines:5
a-symmetric:6
aa:7
ab:8
abbey:9


In [10]:
# load processed review text
df = pd.read_csv('processed.csv')
reviews = df['Processed Review Text'].tolist()

# preview output
print(f"Total reviews: {len(reviews)}")
print(f"Preview:\n{reviews[10]}")

Total reviews: 19662
Preview:
black xs larkspur midi bother lining skirt portion stats xs smoothly chest flowy lower half running big straps pretty easily knees


In [11]:
# checking for NaN values
df.info() # there are 10 NaN values

<class 'pandas.DataFrame'>
RangeIndex: 19662 entries, 0 to 19661
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Clothing ID              19662 non-null  int64
 1   Age                      19662 non-null  int64
 2   Title                    19662 non-null  str  
 3   Review Text              19662 non-null  str  
 4   Rating                   19662 non-null  int64
 5   Recommended IND          19662 non-null  int64
 6   Positive Feedback Count  19662 non-null  int64
 7   Division Name            19662 non-null  str  
 8   Department Name          19662 non-null  str  
 9   Class Name               19662 non-null  str  
 10  Processed Review Text    19652 non-null  str  
dtypes: int64(5), str(6)
memory usage: 10.7 MB


In [12]:
# replace NaN values with empty strings
reviews = [review if isinstance(review, str) else '' for review in reviews]

In [13]:
# generate count vectors using CountVectorizer
cVectorizer = CountVectorizer(analyzer='word', vocabulary=vocab_dict)
count_features = cVectorizer.fit_transform(reviews)

print(f"Shape: {count_features.shape}")
print(f"Expected: ({len(df)}, {len(vocab_dict)})")

# preview output
print("Preview:")
for i in range(3):
    row = count_features.getrow(i) # get row as sparse matrix and find non zero entries
    non_zero_indices = row.indices # get indices and values of non zero entries
    non_zero_values = row.data
    
    sparse_repr = ','.join([f"{idx}:{freq}" for idx, freq in 
                            zip(non_zero_indices, non_zero_values)])
    
    print(f"#{i},{sparse_repr}")

Shape: (19662, 7529)
Expected: (19662, 7529)
Preview:
#0,687:1,1028:1,1716:1,1792:1,2289:1,2481:1,2602:1,2892:2,3010:1,3087:1,3193:1,3258:1,3549:2,3552:1,3832:1,3934:1,4224:2,4234:1,4427:1,4639:2,5260:1,5668:1,6726:1,7092:1,7207:1,7406:1,7520:1,7522:1
#1,1287:1,2284:1,2502:1,2667:1,3403:1,6739:1
#2,86:1,925:1,1988:1,2646:1,3584:1,3595:1,4506:1,5736:2,5924:1,6716:1


#### Unweighted Vector Representation

In [14]:
# load the pretrained FastText model
fasttext_model = api.load('fasttext-wiki-news-subwords-300')
print(f"Vector size: {fasttext_model.vector_size}")

Vector size: 300


In [15]:
# function to get unweighted vector for a review
def get_unweighted(review, model):
    vectors = [model[word] for word in review if word in model] # get vectors for words in the model
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    # get the average
    return np.mean(vectors, axis=0)

In [16]:
# tokenized reviews
tk_reviews = [review.split() for review in reviews]

In [17]:
# generate unweighted vectors
unweighted_features = np.array([get_unweighted(review, fasttext_model) 
                                 for review in tk_reviews])

print(f"Shape: {unweighted_features.shape}")

# preview first review vector (first 10 dimensions)
print("Unweighted vector for review 0:")
print(unweighted_features[0][:10])

Shape: (19662, 300)
Unweighted vector for review 0:
[-0.00418848 -0.00578792  0.01316996  0.00376999 -0.01505092 -0.00523034
 -0.01272889 -0.11266441 -0.00399931 -0.0069355 ]


#### Weighted Vector Representation

In [18]:
# generate TF-IDF vectors
tVectorizer = TfidfVectorizer(analyzer='word', vocabulary=vocab_dict) # initialised the TfidfVectorizer
tfidf_matrix = tVectorizer.fit_transform(reviews) # generate the tfidf vector representation for all reviews

print(f"TF-IDF shape: {tfidf_matrix.shape}")

TF-IDF shape: (19662, 7529)


In [19]:
# function to get weighted vectors for a review
def get_weighted(review, tfidf_row, vocab_dict, model):
    weighted_vectors = []
    
    for word in review:
        if word in model and word in vocab_dict:
            word_idx = vocab_dict[word]
            tfidf_weight = tfidf_row[0, word_idx]
            weighted_vectors.append(model[word] * tfidf_weight)
    
    if len(weighted_vectors) == 0:
        return np.zeros(model.vector_size)
    
    # get the average
    return np.mean(weighted_vectors, axis=0)

In [20]:
# generate TF-IDF weighted vectors 
weighted_features = np.array([get_weighted(tk_reviews[i],
                                                   tfidf_matrix.getrow(i),
                                                   vocab_dict,
                                                   fasttext_model)
                               for i in range(len(reviews))])

print(f"Weighted shape: {weighted_features.shape}")

# preview output
print("TF-IDF vector for review 0:")
print(weighted_features[0][:10])

Weighted shape: (19662, 300)
TF-IDF vector for review 0:
[ 0.00015582 -0.00364494  0.00230363 -0.00032571 -0.00307374 -0.00015215
 -0.00444026 -0.02158724  0.00217164 -0.00352162]


### Saving outputs
Save the count vector representation as per spectification.
- count_vectors.txt

In [21]:
# save with the required format
with open('count_vectors.txt', 'w') as f:
    for i in range(count_features.shape[0]):
        row = count_features.getrow(i)
        non_zero_indices = row.indices
        non_zero_values = row.data
        sparse_repr = ','.join([f"{idx}:{freq}" for idx, freq in 
                                zip(non_zero_indices, non_zero_values)])
        f.write(f"#{i},{sparse_repr}\n")

In [22]:
# verifying saved content 
with open('count_vectors.txt', 'r') as f:
    lines = f.read().splitlines()

print(f"Length ofcount_vectors.txt: {len(lines)}")
print("Preview:")
for line in lines[:3]:
    print(line)

Length ofcount_vectors.txt: 19662
Preview:
#0,687:1,1028:1,1716:1,1792:1,2289:1,2481:1,2602:1,2892:2,3010:1,3087:1,3193:1,3258:1,3549:2,3552:1,3832:1,3934:1,4224:2,4234:1,4427:1,4639:2,5260:1,5668:1,6726:1,7092:1,7207:1,7406:1,7520:1,7522:1
#1,1287:1,2284:1,2502:1,2667:1,3403:1,6739:1
#2,86:1,925:1,1988:1,2646:1,3584:1,3595:1,4506:1,5736:2,5924:1,6716:1


## Task 3. Clothing Review Classification

...... Sections and code blocks on buidling classification models based on different document feature represetations. 
Detailed comparsions and evaluations on different models to answer each question as per specification. 

<span style="color: red"> You might have complex notebook structure in this section, please feel free to create your own notebook structure. </span>

In [23]:
seed = 42
num_folds = 5
kf = KFold(n_splits=num_folds, random_state=seed, shuffle=True)
print(kf)


KFold(n_splits=5, random_state=42, shuffle=True)


In [24]:
# load labels 
sentiment = df['Recommended IND'].tolist()
print(f"Length: {len(sentiment)}")
print(f"1: {sum(sentiment)}, 0: {len(sentiment)-sum(sentiment)}")

Length: 19662
1: 16087, 0: 3575


In [25]:
# function to evaluate performance with logistic regression model
def evaluate(X_train, X_test, y_train, y_test, seed):
    model = LogisticRegression(random_state=seed, max_iter=1000)
    model.fit(X_train, y_train)
    accuracy = model.score(X_test, y_test)
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    return accuracy, f1

In [26]:
# creates a dataframe to store the accuracy and f1 scores in all the folds
cv_df = pd.DataFrame(columns=['count_acc', 'unweighted_acc', 'weighted_acc',
                               'count_f1', 'unweighted_f1', 'weighted_f1'], 
                     index=range(num_folds))

fold = 0
for train_index, test_index in kf.split(list(range(0, len(sentiment)))):
    y_train = [sentiment[i] for i in train_index]
    y_test  = [sentiment[i] for i in test_index]

    acc, f1 = evaluate(count_features[train_index], count_features[test_index], y_train, y_test, seed)
    cv_df.loc[fold, 'count_acc'] = acc
    cv_df.loc[fold, 'count_f1'] = f1

    acc, f1 = evaluate(unweighted_features[train_index], unweighted_features[test_index], y_train, y_test, seed)
    cv_df.loc[fold, 'unweighted_acc'] = acc
    cv_df.loc[fold, 'unweighted_f1'] = f1

    acc, f1 = evaluate(weighted_features[train_index], weighted_features[test_index], y_train, y_test, seed)
    cv_df.loc[fold, 'weighted_acc'] = acc
    cv_df.loc[fold, 'weighted_f1'] = f1

    fold += 1

In [27]:
# results per fold
cv_df

,count_acc,unweighted_acc,weighted_acc,count_f1,unweighted_f1,weighted_f1
0,0.882532,0.840834,0.822782,0.876728,0.79293,0.744029
1,0.874142,0.822782,0.803204,0.867708,0.770492,0.715997
2,0.879451,0.840285,0.825534,0.874066,0.788502,0.747377
3,0.868769,0.834435,0.820193,0.859788,0.784825,0.739629
4,0.871821,0.833164,0.81943,0.864864,0.780686,0.738844


In [28]:
# average across all 5 
cv_df.mean()

count_acc         0.875343
unweighted_acc      0.8343
weighted_acc      0.818229
count_f1          0.868631
unweighted_f1     0.783487
weighted_f1       0.737175
dtype: object

In [29]:
# converting to dataframe for clearer comparison
Q1_comparison = []

# accuracy
Q1_comparison.append({
    'Model': 'Accuracy',
    'Count Vector': cv_df['count_acc'].mean(),
    'Unweighted': cv_df['unweighted_acc'].mean(),
    'Weighted': cv_df['weighted_acc'].mean()
})

# f1
Q1_comparison.append({
    'Model': 'F1',
    'Count Vector': cv_df['count_f1'].mean(),
    'Unweighted': cv_df['unweighted_f1'].mean(),
    'Weighted': cv_df['weighted_f1'].mean()
})

# convert to dataframe 
Q1_comparison_df = pd.DataFrame(Q1_comparison).set_index('Model')

### Title Processing

#### Tokenizing Titles and Lowercasing

In [30]:
def tokenizeTitle(raw_title):
    # cover all words to lowercase
    title = raw_title.lower() 

    # segament into sentences
    sentences = sent_tokenize(title)
    
    # tokenize each sentence
    pattern = r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"
    tokenizer = RegexpTokenizer(pattern) 
    token_lists = [tokenizer.tokenize(sen) for sen in sentences]
    
    # merge them into a list of tokens
    tokenised_title = list(chain.from_iterable(token_lists))
    return tokenised_title

In [31]:
tk_title = [tokenizeTitle(t) for t in df['Title']]

emp = 8
print("Original title:")
print(df['Title'][emp])
print("Tokenized title:")
print(tk_title[emp])

Original title:
Dress looks like it's made of cheap material
Tokenized title:
['dress', 'looks', 'like', "it's", 'made', 'of', 'cheap', 'material']


In [32]:
# checking stats after tokenization
def print_stats(tk_title):
    words = list(chain.from_iterable(tk_title))
    vocab = set(words)
    lexical_diversity = len(vocab)/len(words)
    print("Vocabulary size: ", len(vocab))
    print("Total number of tokens: ", len(words))
    print("Lexical diversity: ", lexical_diversity)
    print("Total number of titles:", len(tk_title))
    lens = [len(title) for title in tk_title]
    print("Average title length:", np.mean(lens))
    print("Maximum title length:", np.max(lens))
    print("Minimum title length:", np.min(lens))
    print("Standard deviation of title length:", np.std(lens))

print_stats(tk_title)

Vocabulary size:  3908
Total number of tokens:  65372
Lexical diversity:  0.059780945970752
Total number of titles: 19662
Average title length: 3.3247889329671447
Maximum title length: 12
Minimum title length: 0
Standard deviation of title length: 1.7795830326849922


### Removing Single Character Words

In [33]:
tk_title_rsw = [[word for word in review if len(word) > 1] for review in tk_title]

# preview output
print("Before removing:")
print(tk_title[emp])
print("After removing:")
print(tk_title_rsw[emp])

Before removing:
['dress', 'looks', 'like', "it's", 'made', 'of', 'cheap', 'material']
After removing:
['dress', 'looks', 'like', "it's", 'made', 'of', 'cheap', 'material']


#### Removing Stop Words

In [34]:
# load stopwords
with open('stopwords_en.txt', 'r') as f:
    stopwords = set(f.read().splitlines())

# remove stopwords based on the list given
tk_title_rstop = [[word for word in review if word not in stopwords] for review in tk_title_rsw]

# preview output
print("Before removing:")
print(tk_title_rsw[emp])
print("After removing:")
print(tk_title_rstop[emp])

Before removing:
['dress', 'looks', 'like', "it's", 'made', 'of', 'cheap', 'material']
After removing:
['dress', 'made', 'cheap', 'material']


#### Remove Words that Only Appears Once

In [35]:
# compute document frequency for each unique word/type
words = list(chain.from_iterable(tk_title_rstop))
term_fd = FreqDist(words)

# find words that appear only once 
lessFreqWords = set(term_fd.hapaxes())
print(f"Length of words appearing only once: {len(lessFreqWords)}")
print("Preview", lessFreqWords)

Length of words appearing only once: 1929
Preview {'eghhh', "la's", 'boogie', 'addition-get', 'exceptional', 'broader', 'moment', 'multi-season', 'prototype', 'itch', 'stunningly', 'boom', 'boxes', 'loose-fit', 'snappy', 'prarie', 'punch', 'finished', 'balloons', 't-back', 'stains', 'planned', 'bodysuit', 'boxt', 'reconstructed', 'maximizes', 'picked', 'snails', 'merino', 'happyinpdx', 'joanna', 'mullet', 'ackerley', 'modification', 'bedtime', 'temperatures', 'deodorant', 'combine', 'thrown', 'designs', 'carrie', 'bum', 'sahm', 'huesca', 'loooooooow', 'lovely-runs', 'aesthetically-pleasing', 'discrepency', 'teeny', 'carefull', 'diagonal', 'challange', 'leave', 'shortcake', 'wayyy', 'off-kilter', 'visual', 'werid', 'necessity', 'breathable', 'bulk', 'vitamin', 'consept', 'upsize', 'ehhhh', 'brought', 'swooned', 'luxuriously', 'ladylike', 'partum', 'shearling-lined', 'petties', 'fixes', 'paisley', 'exchange', 'daughter', 'washers', 'kate', 'anti-jean', 'gadget', 'ack', 'hadley', 'fyi', '

In [36]:
# remove lessFreqWords
tk_title_rhapaxes = [[word for word in review if word not in lessFreqWords] for review in tk_title_rstop]

# preview output
print("Before removing:")
print(tk_title_rstop[emp])
print("After removing:")
print(tk_title_rhapaxes[emp])

Before removing:
['dress', 'made', 'cheap', 'material']
After removing:
['dress', 'made', 'cheap', 'material']


#### Remove Top 20 Most Freq Words

In [37]:
# compute document frequency for each unique word/type
words_df = list(chain.from_iterable([set(review) for review in tk_title_rhapaxes]))
doc_fd = FreqDist(words_df)

# find the top 20 most frequent words 
top20 = set([word for word, freq in doc_fd.most_common(20)])
print("Top 20:")
print(doc_fd.most_common(20))

Top 20:
[('great', 1771), ('love', 1676), ('dress', 1642), ('cute', 1543), ('beautiful', 1399), ('top', 1174), ('perfect', 816), ('pretty', 669), ('fit', 611), ('nice', 527), ('flattering', 504), ('runs', 486), ('comfortable', 472), ('lovely', 471), ('comfy', 455), ('gorgeous', 440), ('summer', 440), ('soft', 434), ('sweater', 424), ('small', 337)]


In [38]:
# remove the top 20 mostFreqWords
tk_title_top20 = [[word for word in review if word not in top20] for review in tk_title_rhapaxes]

# preview output
print("Before removing:")
print(tk_title_rhapaxes[emp])
print("After removing:")
print(tk_title_top20[emp])

Before removing:
['dress', 'made', 'cheap', 'material']
After removing:
['made', 'cheap', 'material']


In [39]:
# comparing stats before and after preprocessing
print("Before preprocessing:")
print_stats(tk_title)
print()
print("After preprocessing:")
print_stats(tk_title_top20)
# vocab size and number of tokens decreased
# lexical diversity increased

Before preprocessing:
Vocabulary size:  3908
Total number of tokens:  65372
Lexical diversity:  0.059780945970752
Total number of titles: 19662
Average title length: 3.3247889329671447
Maximum title length: 12
Minimum title length: 0
Standard deviation of title length: 1.7795830326849922

After preprocessing:
Vocabulary size:  1609
Total number of tokens:  24620
Lexical diversity:  0.06535337124289196
Total number of titles: 19662
Average title length: 1.2521615298545417
Maximum title length: 6
Minimum title length: 0
Standard deviation of title length: 1.007745992450796


In [40]:
# joining tokens into string separated by space
joined_title = [' '.join(review) for review in tk_title_top20]

# preview output
print("Length:", len(joined_title)) # equals to number of titles
print("Preview:", joined_title[:5])


Length: 19662
Preview: ['major design flaws', 'favorite buy', 'shirt', 'petite', 'shimmer fun']


In [41]:
# adding new column 
df['Processed Title'] = joined_title

# preview output
print(df.shape)
print(df.head())

(19662, 12)
   Clothing ID  Age                    Title  \
0         1077   60  Some major design flaws   
1         1049   50         My favorite buy!   
2          847   47         Flattering shirt   
3         1080   49  Not for the very petite   
4          858   39     Cagrcoal shimmer fun   

                                         Review Text  Rating  Recommended IND  \
0  I had such high hopes for this dress and reall...       3                0   
1  I love, love, love this jumpsuit. it's fun, fl...       5                1   
2  This shirt is very flattering to all due to th...       5                1   
3  I love tracy reese dresses, but this one is no...       2                0   
4  I aded this in my basket at hte last mintue to...       5                1   

   Positive Feedback Count   Division Name Department Name Class Name  \
0                        0         General         Dresses    Dresses   
1                        0  General Petite         Bottoms      Pa

In [42]:
# generating the vocabulary
words = list(chain.from_iterable(tk_title_top20)) # put all the tokens in the corpus in a single list
vocab = sorted(list(set(words)))  # compute vocabulary and sort it alphabetically

print(f"Vocab Length: {len(vocab)}") # equals to the vocab size of tk_title_top20
print("Preview:")
print(vocab[:10])

Vocab Length: 1609
Preview:
['a-line', 'ab', 'absolute', 'absolutely', 'accessory', 'add', 'addition', 'adds', 'adorable', 'adorbs']


In [43]:
# index starting from 0
vocab_title = {word: idx for idx, word in enumerate(vocab)}

# preview output
for word, idx in list(vocab_title.items())[:10]:
    print(f"{word}:{idx}")

a-line:0
ab:1
absolute:2
absolutely:3
accessory:4
add:5
addition:6
adds:7
adorable:8
adorbs:9


In [44]:
# checking for NaN values
df.info() # no NaN values found

<class 'pandas.DataFrame'>
RangeIndex: 19662 entries, 0 to 19661
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Clothing ID              19662 non-null  int64
 1   Age                      19662 non-null  int64
 2   Title                    19662 non-null  str  
 3   Review Text              19662 non-null  str  
 4   Rating                   19662 non-null  int64
 5   Recommended IND          19662 non-null  int64
 6   Positive Feedback Count  19662 non-null  int64
 7   Division Name            19662 non-null  str  
 8   Department Name          19662 non-null  str  
 9   Class Name               19662 non-null  str  
 10  Processed Review Text    19652 non-null  str  
 11  Processed Title          19662 non-null  str  
dtypes: int64(5), str(7)
memory usage: 11.0 MB


#### Count Vector (Title)

In [45]:
# generate count vectors using CountVectorizer
cVectorizer_title = CountVectorizer(analyzer='word', vocabulary=vocab_title)
count_features_title = cVectorizer_title.fit_transform(df['Processed Title'])

print(f"Shape: {count_features_title.shape}")
print(f"Expected: ({len(df)}, {len(vocab_title)})")

# preview output
print("Preview:")
for i in range(3):
    row = count_features.getrow(i) # get row as sparse matrix and find non zero entries
    non_zero_indices = row.indices # get indices and values of non zero entries
    non_zero_values = row.data
    
    sparse_repr = ','.join([f"{idx}:{freq}" for idx, freq in 
                            zip(non_zero_indices, non_zero_values)])
    
    print(f"#{i},{sparse_repr}")

Shape: (19662, 1609)
Expected: (19662, 1609)
Preview:
#0,687:1,1028:1,1716:1,1792:1,2289:1,2481:1,2602:1,2892:2,3010:1,3087:1,3193:1,3258:1,3549:2,3552:1,3832:1,3934:1,4224:2,4234:1,4427:1,4639:2,5260:1,5668:1,6726:1,7092:1,7207:1,7406:1,7520:1,7522:1
#1,1287:1,2284:1,2502:1,2667:1,3403:1,6739:1
#2,86:1,925:1,1988:1,2646:1,3584:1,3595:1,4506:1,5736:2,5924:1,6716:1


#### Unweighted Vector (Title)

In [46]:
# function to get unweighted vector for a title
def get_unweighted(title, model):
    vectors_title = [model[word] for word in title if word in model] # get vectors for words in the model
    if len(vectors_title) == 0:
        return np.zeros(model.vector_size)
    # get the average
    return np.mean(vectors_title, axis=0)

In [47]:
# tokenized titles
tk_titles = [title.split() for title in df['Processed Title']]

In [48]:
# generate unweighted vectors
unweighted_features_title = np.array([get_unweighted(title, fasttext_model) 
                                 for title in tk_titles])

print(f"Shape: {unweighted_features_title.shape}")

# preview first title vector (first 10 dimensions)
print("Unweighted vector for title 0:")
print(unweighted_features_title[0][:10])

Shape: (19662, 300)
Unweighted vector for title 0:
[-0.04142933 -0.043141   -0.000411   -0.0080049  -0.050151   -0.0281194
 -0.00907567 -0.11611667 -0.020979    0.0174774 ]


#### Weighted Vector (TF-IDF) for Title

In [49]:
# generate TF-IDF vectors
tVectorizer_title = TfidfVectorizer(analyzer='word', vocabulary=vocab_title) # initialised the TfidfVectorizer
tfidf_title_matrix = tVectorizer_title.fit_transform(df['Processed Title']) # generate the tfidf vector representation for all titles

print(f"TF-IDF shape: {tfidf_title_matrix.shape}")

TF-IDF shape: (19662, 1609)


In [50]:
# function to get weighted vectors for a title
def get_weighted(title, tfidf_row, vocab_title, model):
    weighted_vectors_title = []

    for word in title:
        if word in model and word in vocab_title:
            word_idx = vocab_title[word]
            tfidf_weight = tfidf_row[0, word_idx]
            weighted_vectors_title.append(model[word] * tfidf_weight)

    if len(weighted_vectors_title) == 0:
        return np.zeros(model.vector_size)
    
    # get the average
    return np.mean(weighted_vectors_title, axis=0)

In [51]:
# generate TF-IDF weighted vectors 
weighted_features_title = np.array([get_weighted(tk_titles[i],
                                                   tfidf_title_matrix.getrow(i),
                                                   vocab_title,
                                                   fasttext_model)
                               for i in range(len(df['Processed Title']))])

print(f"Weighted shape: {weighted_features_title.shape}")

# preview output
print("TF-IDF vector for title 0:")
print(weighted_features_title[0][:10])

Weighted shape: (19662, 300)
TF-IDF vector for title 0:
[-0.02380699 -0.02535376 -0.00012839 -0.00452057 -0.02999983 -0.01318747
 -0.00375747 -0.06691564 -0.01042756  0.00854503]


#### Performance Classification (Title)

In [52]:
# creates a dataframe to store the accuracy and f1 scores in all the folds
cv_title_df = pd.DataFrame(columns=['count_acc_title', 'unweighted_acc_title', 'weighted_acc_title',
                               'count_f1_title', 'unweighted_f1_title', 'weighted_f1_title'], 
                     index=range(num_folds))

fold = 0
for train_index, test_index in kf.split(list(range(0, len(sentiment)))):
    y_train = [sentiment[i] for i in train_index]
    y_test  = [sentiment[i] for i in test_index]

    acc, f1 = evaluate(count_features_title[train_index], count_features_title[test_index], y_train, y_test, seed)
    cv_title_df.loc[fold, 'count_acc_title'] = acc
    cv_title_df.loc[fold, 'count_f1_title'] = f1

    acc, f1 = evaluate(unweighted_features_title[train_index], unweighted_features_title[test_index], y_train, y_test, seed)
    cv_title_df.loc[fold, 'unweighted_acc_title'] = acc
    cv_title_df.loc[fold, 'unweighted_f1_title'] = f1

    acc, f1 = evaluate(weighted_features_title[train_index], weighted_features_title[test_index], y_train, y_test, seed)
    cv_title_df.loc[fold, 'weighted_acc_title'] = acc
    cv_title_df.loc[fold, 'weighted_f1_title'] = f1

    fold += 1

In [53]:
# results per fold
cv_title_df

,count_acc_title,unweighted_acc_title,weighted_acc_title,count_f1_title,unweighted_f1_title,weighted_f1_title
0,0.86626,0.85431,0.844139,0.848003,0.831652,0.814347
1,0.856344,0.843377,0.833715,0.835128,0.817437,0.800834
2,0.873347,0.863428,0.856307,0.855965,0.842527,0.827537
3,0.860885,0.855799,0.847152,0.841574,0.833973,0.817467
4,0.863174,0.849949,0.843591,0.845833,0.827292,0.814071


In [54]:
# average across all 5 
cv_title_df.mean()

count_acc_title         0.864002
unweighted_acc_title    0.853372
weighted_acc_title      0.844981
count_f1_title          0.845301
unweighted_f1_title     0.830576
weighted_f1_title       0.814851
dtype: object

### Title + Review Processing

#### Combining Processed Review Text and Processed Title

In [55]:
df['Processed Title + Review'] = df['Processed Title'] + ' ' + df['Processed Review Text']

# preview output
print(df['Processed Title + Review'][10])

big black xs larkspur midi bother lining skirt portion stats xs smoothly chest flowy lower half running big straps pretty easily knees


In [56]:
# checking for NaN values
df.info() # there are 10 NaN values from the Reviews

<class 'pandas.DataFrame'>
RangeIndex: 19662 entries, 0 to 19661
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Clothing ID               19662 non-null  int64
 1   Age                       19662 non-null  int64
 2   Title                     19662 non-null  str  
 3   Review Text               19662 non-null  str  
 4   Rating                    19662 non-null  int64
 5   Recommended IND           19662 non-null  int64
 6   Positive Feedback Count   19662 non-null  int64
 7   Division Name             19662 non-null  str  
 8   Department Name           19662 non-null  str  
 9   Class Name                19662 non-null  str  
 10  Processed Review Text     19652 non-null  str  
 11  Processed Title           19662 non-null  str  
 12  Processed Title + Review  19652 non-null  str  
dtypes: int64(5), str(8)
memory usage: 13.6 MB


In [57]:
# handle NaN values by inserting blank strings
df['Processed Title + Review'] = df['Processed Title + Review'].fillna('')

In [58]:
# checking out the updated df
df.head()

,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name,Processed Review Text,Processed Title,Processed Title + Review
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses,high hopes wanted work initially petite usual ...,major design flaws,major design flaws high hopes wanted work init...
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants,jumpsuit fun flirty fabulous time compliments,favorite buy,favorite buy jumpsuit fun flirty fabulous time...
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses,shirt due adjustable front tie length leggings...,shirt,shirt shirt due adjustable front tie length le...
3,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",2,0,4,General,Dresses,Dresses,tracy reese dresses petite feet tall brand pre...,petite,petite tracy reese dresses petite feet tall br...
4,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,5,1,1,General Petite,Tops,Knits,basket hte person store pick teh pale hte gorg...,shimmer fun,shimmer fun basket hte person store pick teh p...


#### Conbining Vocab

In [59]:
# create combined vocabulary by copying the content of vocab_dict
vocab_combined = vocab_dict.copy()

# add title vocab if its not already in vocab_combined
next_idx = len(vocab_combined)
for word in vocab_title:
    if word not in vocab_combined:
        vocab_combined[word] = next_idx
        next_idx += 1

print(f"Review vocab size: {len(vocab_dict)}")
print(f"Title vocab size: {len(vocab_title)}")
print(f"Combined vocab size: {len(vocab_combined)}")

Review vocab size: 7529
Title vocab size: 1609
Combined vocab size: 7617


#### Count Vector (Title + Review)

In [60]:
# generate count vectors using CountVectorizer
cVectorizer_combined = CountVectorizer(analyzer='word', vocabulary=vocab_combined)
count_features_combined = cVectorizer.fit_transform(df['Processed Title + Review'])

print(f"Shape: {count_features_combined.shape}")
print(f"Expected: ({len(df)}, {len(vocab_combined)})")

# preview output
print("Preview:")
for i in range(3):
    row = count_features_combined.getrow(i) # get row as sparse matrix and find non zero entries
    non_zero_indices = row.indices # get indices and values of non zero entries
    non_zero_values = row.data
    
    sparse_repr = ','.join([f"{idx}:{freq}" for idx, freq in 
                            zip(non_zero_indices, non_zero_values)])
    
    print(f"#{i},{sparse_repr}")

Shape: (19662, 7529)
Expected: (19662, 7617)
Preview:
#0,687:1,1028:1,1716:2,1792:1,2289:1,2481:1,2485:1,2602:1,2892:2,3010:1,3087:1,3193:1,3258:1,3549:2,3552:1,3832:2,3934:1,4224:2,4234:1,4427:1,4639:2,5260:1,5668:1,6726:1,7092:1,7207:1,7406:1,7520:1,7522:1
#1,868:1,1287:1,2284:1,2347:1,2502:1,2667:1,3403:1,6739:1
#2,86:1,925:1,1988:1,2646:1,3584:1,3595:1,4506:1,5736:3,5924:1,6716:1


#### Unweighted Vector (Title + Review)

In [61]:
# function to get unweighted vector 
def get_unweighted(text, model):
    vectors_combined = [model[word] for word in text if word in model] # get vectors for words in the model
    if len(vectors_combined) == 0:
        return np.zeros(model.vector_size)
    # get the average
    return np.mean(vectors_combined, axis=0)

In [62]:
# tokenizing
tk_combined = [text.split() for text in df['Processed Title + Review']]

In [63]:
# generate unweighted vectors
unweighted_features_combined = np.array([get_unweighted(text, fasttext_model) 
                                 for text in tk_combined])

print(f"Shape: {unweighted_features_combined.shape}")

# preview first combined vector (first 10 dimensions)
print("Unweighted vector for combined text 0:")
print(unweighted_features_combined[0][:10])

Shape: (19662, 300)
Unweighted vector for combined text 0:
[-0.00738055 -0.00898961  0.01200588  0.00276072 -0.0180595  -0.00719226
 -0.01241576 -0.11296032 -0.00545471 -0.00484296]


#### Weighted Vector (Title + Review)

In [64]:
# generate TF-IDF vectors
tVectorizer_combined = TfidfVectorizer(analyzer='word', vocabulary=vocab_combined) # initialised the TfidfVectorizer
tfidf_combined_matrix = tVectorizer_combined.fit_transform(df['Processed Title + Review']) # generate the tfidf vector representation for all text

print(f"TF-IDF shape: {tfidf_combined_matrix.shape}")

TF-IDF shape: (19662, 7617)


In [65]:
# function to get weighted vectors
def get_weighted(text, tfidf_row, vocab_combined, model):
    weighted_vectors_combined = []

    for word in text:
        if word in model and word in vocab_combined:
            word_idx = vocab_combined[word]
            tfidf_weight = tfidf_row[0, word_idx]
            weighted_vectors_combined.append(model[word] * tfidf_weight)

    if len(weighted_vectors_combined) == 0:
        return np.zeros(model.vector_size)
    
    # get the average
    return np.mean(weighted_vectors_combined, axis=0)

In [66]:
# generate TF-IDF weighted vectors 
weighted_features_combined = np.array([get_weighted(tk_combined[i],
                                                   tfidf_combined_matrix.getrow(i),
                                                   vocab_combined,
                                                   fasttext_model)
                               for i in range(len(df['Processed Title + Review']))])

print(f"Weighted shape: {weighted_features_combined.shape}")

# preview output
print("TF-IDF vector for combined text 0:")
print(weighted_features_combined[0][:10])

Weighted shape: (19662, 300)
TF-IDF vector for combined text 0:
[-0.00082688 -0.00457154  0.00147514 -0.00056174 -0.00392019 -0.00072473
 -0.00382935 -0.021533    0.00164632 -0.00265143]


#### Performance Classification (Title + Review)

In [67]:
# creates a dataframe to store the accuracy and f1 scores in all the folds
cv_combined_df = pd.DataFrame(columns=['count_acc_combined', 'unweighted_acc_combined', 'weighted_acc_combined',
                               'count_f1_combined', 'unweighted_f1_combined', 'weighted_f1_combined'], 
                     index=range(num_folds))

fold = 0
for train_index, test_index in kf.split(list(range(0, len(sentiment)))):
    y_train = [sentiment[i] for i in train_index]
    y_test  = [sentiment[i] for i in test_index]

    acc, f1 = evaluate(count_features_combined[train_index], count_features_combined[test_index], y_train, y_test, seed)
    cv_combined_df.loc[fold, 'count_acc_combined'] = acc
    cv_combined_df.loc[fold, 'count_f1_combined'] = f1

    acc, f1 = evaluate(unweighted_features_combined[train_index], unweighted_features_combined[test_index], y_train, y_test, seed)
    cv_combined_df.loc[fold, 'unweighted_acc_combined'] = acc
    cv_combined_df.loc[fold, 'unweighted_f1_combined'] = f1

    acc, f1 = evaluate(weighted_features_combined[train_index], weighted_features_combined[test_index], y_train, y_test, seed)
    cv_combined_df.loc[fold, 'weighted_acc_combined'] = acc
    cv_combined_df.loc[fold, 'weighted_f1_combined'] = f1

    fold += 1

In [68]:
# result per fold
cv_combined_df

,count_acc_combined,unweighted_acc_combined,weighted_acc_combined,count_f1_combined,unweighted_f1_combined,weighted_f1_combined
0,0.896517,0.851259,0.822527,0.89279,0.811714,0.743423
1,0.886855,0.830409,0.803966,0.88226,0.78427,0.717338
2,0.898016,0.848678,0.825788,0.894681,0.808087,0.747504
3,0.880214,0.843082,0.820956,0.875099,0.801915,0.742406
4,0.886826,0.846134,0.819939,0.881728,0.805888,0.73958


In [69]:
# average result across all 5
cv_combined_df.mean()

count_acc_combined         0.889685
unweighted_acc_combined    0.843912
weighted_acc_combined      0.818635
count_f1_combined          0.885312
unweighted_f1_combined     0.802375
weighted_f1_combined        0.73805
dtype: object

#### Comparison of Result

In [70]:
acc_comparison = []

# Bag of Words (Count Vector)
acc_comparison.append({
    'Model': 'Count Vector',
    'Title': cv_title_df['count_acc_title'].mean(),
    'Review Text': cv_df['count_acc'].mean(),
    'Combined': cv_combined_df['count_acc_combined'].mean()
})

# Unweighted
acc_comparison.append({
    'Model': 'Unweighted Vector',
    'Title': cv_title_df['unweighted_acc_title'].mean(),
    'Review Text': cv_df['unweighted_acc'].mean(),
    'Combined': cv_combined_df['unweighted_acc_combined'].mean()
})

# Weighted TF-IDF
acc_comparison.append({
    'Model': 'Weighted TF-IDF',
    'Title': cv_title_df['weighted_acc_title'].mean(),
    'Review Text': cv_df['weighted_acc'].mean(),
    'Combined': cv_combined_df['weighted_acc_combined'].mean()
})

# convert to dataframe 
acc_comparison_df = pd.DataFrame(acc_comparison).set_index('Model')


In [71]:
f1_comparison = []

# Bag of Words (Count Vector)
f1_comparison.append({
    'Model': 'Count Vector',
    'Title': cv_title_df['count_f1_title'].mean(),
    'Review Text': cv_df['count_f1'].mean(),
    'Combined': cv_combined_df['count_f1_combined'].mean()
})

# Unweighted
f1_comparison.append({
    'Model': 'Unweighted Vector',
    'Title': cv_title_df['unweighted_f1_title'].mean(),
    'Review Text': cv_df['unweighted_f1'].mean(),
    'Combined': cv_combined_df['unweighted_f1_combined'].mean()
})

# Weighted TF-IDF
f1_comparison.append({
    'Model': 'Weighted TF-IDF',
    'Title': cv_title_df['weighted_f1_title'].mean(),
    'Review Text': cv_df['weighted_f1'].mean(),
    'Combined': cv_combined_df['weighted_f1_combined'].mean()
})

# convert to dataframe 
f1_comparison_df = pd.DataFrame(f1_comparison).set_index('Model')


## Summary

Q1: Which language model we built previously (based on clothing reviews) performs the best with the chosen machine learning model? 

Ans: Count Vector performs best with accuracy of 87.5% and f1 of 86.8%. Likely because the vocabulary was built specifically from the clothing review corpus, making word frequencies highly discriminative for this domain. FastText embeddings, being pretrained on general Wikipedia text, may not capture clothing-specific language as effectively.

In [72]:
print(Q1_comparison_df)


          Count Vector  Unweighted  Weighted
Model                                       
Accuracy      0.875343    0.834300  0.818229
F1            0.868631    0.783487  0.737175


Q2: Does more information provide higher accuracy?

Ans: Yes, overall, adding more information gives higher accuracy. The count vector for combination gives 88.9% for accuracy and 88.5% for f1, being the highest of the three. However, Review Text alone surprisingly underperforms Title alone for Unweighted and Weighted models, suggesting that longer text does not always guarantee better performance with embedding-based representations. Count Vector remains the exception, where Review Text outperforms Title, likely because its domain-specific vocabulary better captures review text patterns.

In [73]:
print("Accuracy Comparison:")
print(acc_comparison_df)
print("\nF1 Comparison:")
print(f1_comparison_df)

Accuracy Comparison:
                      Title  Review Text  Combined
Model                                             
Count Vector       0.864002     0.875343  0.889685
Unweighted Vector  0.853372     0.834300  0.843912
Weighted TF-IDF    0.844981     0.818229  0.818635

F1 Comparison:
                      Title  Review Text  Combined
Model                                             
Count Vector       0.845301     0.868631  0.885312
Unweighted Vector  0.830576     0.783487  0.802375
Weighted TF-IDF    0.814851     0.737175  0.738050


In [74]:
# ============================================================
# Saving model for Milestone 2
# Train on full dataset (no train/test split) using Count Vector
# since it performed best in Task 3
# ============================================================

import pickle

# retrain on full dataset
final_model = LogisticRegression(random_state=seed, max_iter=1000)
final_model.fit(count_features, sentiment)

# save the trained logistic regression model
with open('logistic_regression_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

# save the count vectorizer (contains vocab_dict already)
with open('count_vectorizer.pkl', 'wb') as f:
    pickle.dump(cVectorizer, f)

print("Model saved: logistic_regression_model.pkl")
print("Vectorizer saved: count_vectorizer.pkl")

Model saved: logistic_regression_model.pkl
Vectorizer saved: count_vectorizer.pkl
